# UNC Baseball Midseason Scoring

This notebook rebuilds the current UNC baseball master feature set, loads the frozen 2026 day-before artifact, rescoring current 2026 rows, and overwrites the three midseason CSVs the app reads.

In [1]:
import numpy as np
import pandas as pd
import requests
import re
import joblib
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.max_info_columns", 200)

In [2]:
# Assumption: notebook is run from PROJECT_ROOT/SCORING/notebooks/
NB_DIR = Path.cwd().resolve()
SCORING_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
PROJECT_ROOT = SCORING_ROOT.parent
SCRAPERS_ROOT = PROJECT_ROOT / "SCRAPERS"
APP_ROOT = PROJECT_ROOT / "APP"

DATA_ROOT = SCORING_ROOT / "data"
TEMPLATES_DIR = SCORING_ROOT / "templates"
ARTIFACTS_DIR = SCORING_ROOT / "artifacts"
APP_DATA_DIR = APP_ROOT / "data"

APP_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

UNC_BASEBALL_CSV = SCRAPERS_ROOT / "GoHeels_Attendance_Scraper" / "unc_baseball_final.csv"
PERFORMANCE_CSV = SCRAPERS_ROOT / "Game_Stat_Scraper" / "exports" / "uncbaseball_season_performance_0331.csv"

ACC_MAPPING_BASEBALL_CSV = DATA_ROOT / "acc_mapping_baseball.csv"
ACADEMIC_CALENDAR_CSV = DATA_ROOT / "academic_calendar_data.csv"
ENROLLMENT_CSV = DATA_ROOT / "enrollment_data.csv"
GTRENDS_WEB_CSV = DATA_ROOT / "gtrends_data_web_uncbaseball.csv"
GTRENDS_YT_CSV = DATA_ROOT / "gtrends_data_yt_uncbaseball.csv"
UNC_BASKETBALL_CSV = DATA_ROOT / "unc_basketball_final.csv"
TEAM_HOME_COORDS_BASEBALL_CSV = DATA_ROOT / "team_home_coords_baseball.csv"

DB_TEMPLATE_FULL = TEMPLATES_DIR / "db_predictions_2024_2026_full_df.csv"
DB_TEMPLATE_RAW = TEMPLATES_DIR / "db_predictions_2024_2026_tier5_raw_tier_cols.csv"
DB_TEMPLATE_FINAL = TEMPLATES_DIR / "db_predictions_2024_2026_tier5_final_feats_export.csv"

DB_OUTPUT_FULL = APP_DATA_DIR / "db_predictions_2024_2026_full_df.csv"
DB_OUTPUT_RAW = APP_DATA_DIR / "db_predictions_2024_2026_tier5_raw_tier_cols.csv"
DB_OUTPUT_FINAL = APP_DATA_DIR / "db_predictions_2024_2026_tier5_final_feats_export.csv"

DB_ARTIFACT_PATH = ARTIFACTS_DIR / "db_frozen_model_for_2026.joblib"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRAPERS_ROOT:", SCRAPERS_ROOT)
print("SCORING_ROOT:", SCORING_ROOT)
print("APP_ROOT:", APP_ROOT)
print("APP_DATA_DIR:", APP_DATA_DIR)
print("UNC_BASEBALL_CSV:", UNC_BASEBALL_CSV)
print("PERFORMANCE_CSV:", PERFORMANCE_CSV)
print("DB_ARTIFACT_PATH:", DB_ARTIFACT_PATH)


PROJECT_ROOT: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-
SCRAPERS_ROOT: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/SCRAPERS
SCORING_ROOT: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/SCORING
APP_ROOT: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/APP
APP_DATA_DIR: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/APP/data
UNC_BASEBALL_CSV: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/SCRAPERS/GoHeels_Attendance_Scraper/unc_baseball_final.csv
PERFORMANCE_CSV: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/SCRAPERS/Game_Stat_Scraper/exports/uncbaseball_season_performance_0331.csv
DB_ARTIFACT_PATH: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/SCORING/artifacts/db_frozen_model_for_2026.joblib


In [3]:
scrape = pd.read_csv(UNC_BASEBALL_CSV)

In [4]:
### FILTER OUT INCOMPLETE ROWS
# drop rows where key fields are missing or game wasn't actually played
latest_season = scrape["season"].max()

scrape = scrape[
    scrape["date"].notna() &
    scrape["opponent"].notna() &
    (
        scrape["attendance"].notna() |  # keep rows with attendance
        (scrape["season"] == latest_season)  # OR keep latest season regardless
    )
].copy()



In [5]:
scrape.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1657 entries, 0 to 1713
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   season          1657 non-null   int64  
 1   game_number     1657 non-null   int64  
 2   date            1657 non-null   object 
 3   start_time      1656 non-null   object 
 4   opponent        1657 non-null   object 
 5   tournament      211 non-null    object 
 6   venue           1657 non-null   object 
 7   promotion       130 non-null    object 
 8   result          1634 non-null   object 
 9   unc_score       1634 non-null   float64
 10  opponent_score  1634 non-null   float64
 11  attendance      1634 non-null   float64
dtypes: float64(3), int64(2), object(7)
memory usage: 168.3+ KB


In [6]:
### NORMALIZE DATES
s = scrape.copy()

# date normalize
mon_day = s["date"].astype(str).str.extract(r"([A-Za-z]{3,9})\s+(\d{1,2})")
s["month_str"] = mon_day[0]
s["day_num"] = pd.to_numeric(mon_day[1], errors="coerce")

s["date_norm"] = pd.to_datetime(
    s["season"].astype(str) + "-" + s["month_str"] + "-" + s["day_num"].astype("Int64").astype(str),
    errors="coerce"
)

# time normalize
t = s["start_time"].astype(str).str.strip()
parsed_time = pd.to_datetime(t, format="%I:%M %p", errors="coerce")
s["time_norm"] = parsed_time.dt.strftime("%H:%M")

# exact_time
s["exact_time"] = pd.to_datetime(
    s["date_norm"].dt.strftime("%Y-%m-%d") + " " + s["time_norm"],
    errors="coerce"
)

### Features

In [7]:
### PARSE WIN/LOSS AND SCORES

scrape = s.copy()

# extract result (W/L/T) and scores from result string
scrape["result"] = scrape["result"].astype("string").str.strip()
parsed = scrape["result"].str.extract(r"^\s*([WLT])\s*,\s*(\d+)\s*-\s*(\d+)\s*$")
scrape["win"] = parsed[0].map({"W": 1, "L": 0, "T": 0.5})
scrape = scrape.drop(columns=["result"])

In [8]:
### GROUPED OPPONENT FOR MODELING
# bucket rare opponents into "Other" so the model doesn't overfit on one-off teams

MIN_GAMES = 10
counts = scrape["opponent"].value_counts()
common_opponents = counts[counts >= MIN_GAMES].index
scrape["opponent_grp"] = scrape["opponent"].where(
    scrape["opponent"].isin(common_opponents),
    "Other"
)

In [9]:
### RECENT FORM (win streaks, rolling wins, record pct)
scrape = scrape.sort_values(["season", "date_norm", "exact_time"]).reset_index(drop=True).copy()

g = scrape.groupby("season")

def add_recent_form(df_season: pd.DataFrame) -> pd.DataFrame:
    w = df_season["win"].fillna(0).astype(int)
    played = df_season["win"].notna().astype(int)

    # prior scored game win
    df_season["winlast"] = df_season["win"].shift(1)

    # rolling wins
    wins3  = w.shift(1).rolling(3, min_periods=1).sum()
    games3 = played.shift(1).rolling(3, min_periods=1).sum()

    wins5  = w.shift(1).rolling(5, min_periods=1).sum()
    games5 = played.shift(1).rolling(5, min_periods=1).sum()

    # store counts
    df_season["last3games"] = wins3.fillna(0).astype(int)

    df_season["last5games"] = wins5.fillna(0).astype(int)

    # record entering game
    df_season["wins_entering"] = w.cumsum().shift(1).fillna(0).astype(int)
    df_season["games_entering"] = played.cumsum().shift(1).fillna(0).astype(int)
    df_season["record_pct"] = (df_season["wins_entering"] / df_season["games_entering"]).replace([np.inf], np.nan)

    return df_season

scrape = g.apply(add_recent_form).reset_index(drop=True)

# previous season win% (static)
season_summary = (
    scrape.groupby("season")["win"]
         .agg(wins=lambda s: s.sum(skipna=True),
              games=lambda s: s.notna().sum())
)
season_summary["win_pct_final"] = season_summary["wins"] / season_summary["games"]
scrape["prev_szn_record_pct"] = scrape["season"].map(season_summary["win_pct_final"].shift(1))


/var/folders/pt/y9n6c10x2kz3_ty0xm4p0gnm0000gn/T/ipykernel_37174/2782670416.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scrape = g.apply(add_recent_form).reset_index(drop=True)


In [10]:
### DROP COLUMNS WE DON'T NEED ANYMORE
# these are either raw scrape fields or columns we've already used
cols_to_drop = [
    "home_away",
    "radio",
    "photos_link",
    "video_link",
    "preview_link",
    "start_time",
    "month_str",
    "venue_clean",
    "opponent_clean",
    "wins_entering",
    "games_entering"
]

scrape = scrape.drop(columns=[c for c in cols_to_drop if c in scrape.columns])

In [11]:
### SEASON STAGE
t = scrape["tournament"].astype(str).str.lower()

scrape["season_stage"] = np.select(
    [
        t.str.contains("super regional", na=False),
        t.str.contains("regional", na=False),
    ],
    [
        "super_regional",
        "regional",
    ],
    default="regular"
)
scrape = scrape.drop(columns=["tournament"])

### Append ACC Map

In [12]:
acc = pd.read_csv(ACC_MAPPING_BASEBALL_CSV)
acc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   season  347 non-null    int64 
 1   team    347 non-null    object
dtypes: int64(1), object(1)
memory usage: 5.6+ KB


In [13]:
### BUILD ACC MEMBERSHIP FLAG
# acc has columns: season (int), team (str)
acc_teams_by_season = acc.groupby("season")["team"].apply(set).to_dict()

# check if opponent was in the ACC that season
scrape["acc"] = scrape.apply(
    lambda r: int(r["opponent"] in acc_teams_by_season.get(r["season"], set())),
    axis=1
)

### More features (home only now)

In [14]:
### HOME GAMES ONLY (Boshamer Stadium filter)

s = scrape.copy()

s["venue_clean"] = s["venue"].astype(str).str.lower().str.strip()
home_mask = s["venue_clean"].str.startswith("bosh")
scrape = s.loc[home_mask].copy()

scrape.info()

<class 'pandas.core.frame.DataFrame'>
Index: 982 entries, 3 to 1651
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   season               982 non-null    int64         
 1   game_number          982 non-null    int64         
 2   date                 982 non-null    object        
 3   opponent             982 non-null    object        
 4   venue                982 non-null    object        
 5   promotion            129 non-null    object        
 6   unc_score            970 non-null    float64       
 7   opponent_score       970 non-null    float64       
 8   attendance           970 non-null    float64       
 9   day_num              982 non-null    int64         
 10  date_norm            982 non-null    datetime64[ns]
 11  time_norm            981 non-null    object        
 12  exact_time           981 non-null    datetime64[ns]
 13  win                  970 non-null    fl

In [15]:
### SERIES DETECTION

scrape = scrape.copy()

scrape = scrape.sort_values(["opponent", "date_norm", "exact_time"]).reset_index(drop=True)

scrape["day_gap"] = (
    scrape.groupby("opponent")["date_norm"]
         .diff()
         .dt.days
)

# New series if first row for opponent or gap > 2 days
scrape["new_series"] = scrape["day_gap"].isna() | (scrape["day_gap"] > 2)

scrape["series_id"] = (
    scrape.groupby("opponent")["new_series"]
         .cumsum()
         .astype(int)
)

# SERIES DETECTION size
scrape["series_size"] = (
    scrape.groupby(["opponent", "series_id"])["date_norm"]
         .transform("size")
)

# SERIES DETECTION flag
scrape["series"] = (scrape["series_size"] > 1).astype(int)

# Game in Series
scrape["game_in_series"] = (
    scrape.groupby(["opponent", "series_id"])
         .cumcount()
         .add(1)
)

# Non-series games = 0 (Maybe this should be null)
scrape.loc[scrape["series"].eq(0), "game_in_series"] = 0
scrape["game_in_series"] = scrape["game_in_series"].fillna(0).astype(int)

# Cleanup
scrape = scrape.drop(columns=["day_gap", "new_series", "series_id", "series_size"])



In [16]:
### DAYS SINCE LAST HOME GAME
scrape = scrape.sort_values(["season", "date_norm", "exact_time"]).copy()

scrape["days_since_last_hg"] = (
    scrape.groupby("season")["date_norm"]
    .diff()
    .dt.days
)

scrape["days_since_last_hg"] = scrape["days_since_last_hg"].fillna(30)


In [17]:
### OPENING DAY FLAG
scrape = scrape.sort_values(["season", "date_norm", "exact_time"])

first_game_idx = (
    scrape
    .dropna(subset=["date_norm"])
    .groupby("season", as_index=False)
    .head(1)
    .index
)

scrape["opening_day"] = 0
scrape.loc[first_game_idx, "opening_day"] = 1


In [18]:
### DAY OF THE WEEK
scrape["day_of_week"] = scrape["exact_time"].dt.day_name()

In [19]:
### DROP MORE RAW COLUMNS AFTER HOME FILTERING
# these were only needed during data cleaning
cols_to_drop = [
    "home_away",
    "date",
    "radio",
    "photos_link",
    "video_link",
    "preview_link",
    "start_time",
    "month_str",
    "venue_clean",
    "location_city_clean",
    "opponent_clean",
    "wins_entering",
    "games_entering",
    "location_city",
    "venue",
    "day_num"
]

scrape = scrape.drop(columns=[c for c in cols_to_drop if c in scrape.columns])

In [20]:
### RENAME COLUMNS
scrape = scrape.rename(columns={
    "weather": "weather_scrape",
    "promotion": "promo_scrape",
    "date_norm": "date",
    "time_norm": "time",
    "opponent_score": "opp_score"
})

In [21]:
scrape.info()

<class 'pandas.core.frame.DataFrame'>
Index: 982 entries, 653 to 610
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   season               982 non-null    int64         
 1   game_number          982 non-null    int64         
 2   opponent             982 non-null    object        
 3   promo_scrape         129 non-null    object        
 4   unc_score            970 non-null    float64       
 5   opp_score            970 non-null    float64       
 6   attendance           970 non-null    float64       
 7   date                 982 non-null    datetime64[ns]
 8   time                 981 non-null    object        
 9   exact_time           981 non-null    datetime64[ns]
 10  win                  970 non-null    float64       
 11  opponent_grp         982 non-null    object        
 12  winlast              949 non-null    float64       
 13  last3games           982 non-null    i

In [22]:
### CREATE MASTER DATAFRAME AND EVENT ID
df = scrape.copy()

# unique event id from datetime + cleaned opponent name
df["event_id"] = (
    df["exact_time"].dt.strftime("%Y%m%d_%H%M") + "_" +
    df["opponent"].str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
)

In [23]:
### WEATHER DATA (Open-Meteo anti-leakage proxy)

import numpy as np
import pandas as pd
import requests

LAT, LON = 35.9115, -79.0528  # The Bosh
TIMEZONE = "America/New_York"

# Choose prediction horizon:
FORECAST_LEAD_DAYS = 2

# prefix
PREFIX = f"wx_d{FORECAST_LEAD_DAYS}"

games = df.copy()
games["exact_time"] = pd.to_datetime(games["exact_time"])
games = games.sort_values("exact_time").reset_index(drop=True)

# Drop rows with no valid datetime - unplayed 2026 games get NaN weather via the left merge at the end
games = games.dropna(subset=["exact_time"])

start_date = (games["exact_time"].min() - pd.Timedelta(days=FORECAST_LEAD_DAYS + 7)).strftime("%Y-%m-%d")

end_date = min(
    games["exact_time"].max().normalize(),
    pd.Timestamp.today().normalize()
).strftime("%Y-%m-%d")

# Open-Meteo archive API
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation",
        "rain",
        "snowfall",
        "cloud_cover",
        "weather_code"
    ],
    "timezone": TIMEZONE
}

r = requests.get(url, params=params, timeout=60)
r.raise_for_status()
data = r.json()

wx = pd.DataFrame({
    "wx_time": pd.to_datetime(data["hourly"]["time"]),
    "temp_c": data["hourly"]["temperature_2m"],
    "humidity": data["hourly"]["relative_humidity_2m"],
    "wind_kmh": data["hourly"]["wind_speed_10m"],
    "precip_mm": data["hourly"]["precipitation"],
    "rain_mm": data["hourly"]["rain"],
    "snowfall_cm": data["hourly"]["snowfall"],
    "cloud_cover": data["hourly"]["cloud_cover"],
    "weathercode": data["hourly"]["weather_code"]
}).sort_values("wx_time").reset_index(drop=True)

# Helpers

def weather_condition_from_code(code):
    if pd.isna(code):
        return np.nan

    code = int(code)

    if code == 0:
        return "clear"
    if code in [1, 2, 3]:
        return "cloudy"
    if code in [45, 48]:
        return "foggy"
    if code in [51, 53, 55, 56, 57]:
        return "rainy"
    if code in [61, 63, 65, 66, 67]:
        return "rainy"
    if code in [71, 73, 75, 77]:
        return "snowy"
    if code in [80, 81, 82]:
        return "rainy"
    if code in [85, 86]:
        return "snowy"
    if code in [95, 96, 99]:
        return "stormy"
    return "cloudy"

def precip_type(row, precip_col="precip_mm", rain_col="rain_mm", snow_col="snowfall_cm"):
    precip = row.get(precip_col, np.nan)
    if pd.isna(precip) or precip == 0:
        return "none"

    rain = row.get(rain_col, 0)
    snow_cm = row.get(snow_col, 0)

    rain = 0 if pd.isna(rain) else rain
    snow_cm = 0 if pd.isna(snow_cm) else snow_cm

    if snow_cm > 0 and rain > 0:
        return "mixed"
    if snow_cm > 0:
        return "snow"
    return "rain"

def comfort_index(temp_c, rh_percent, wind_kmh):
    if pd.isna(temp_c) or pd.isna(rh_percent) or pd.isna(wind_kmh):
        return np.nan
    F = rh_percent / 100.0
    Ws = wind_kmh
    T = temp_c
    return 1.8 * T - 0.55 * (1.8 * T - 26) * (1 - F) - 3.2 * np.sqrt(Ws) + 32

# Derived weather fields

wx["temperature_f"] = wx["temp_c"] * 9 / 5 + 32
wx["wind_mph"] = wx["wind_kmh"] * 0.621371
wx["weather_condition"] = wx["weathercode"].apply(weather_condition_from_code)
wx["precipitation_type"] = wx.apply(precip_type, axis=1)
wx["comfort_index"] = [
    comfort_index(t, rh, ws)
    for t, rh, ws in zip(wx["temp_c"], wx["humidity"], wx["wind_kmh"])
]

# Rolling features using only past weather
wx = wx.sort_values("wx_time").reset_index(drop=True)

# previous 24h means
wx["temp_f_avg_prev24h"] = wx["temperature_f"].shift(1).rolling(24, min_periods=6).mean()
wx["wind_mph_avg_prev24h"] = wx["wind_mph"].shift(1).rolling(24, min_periods=6).mean()
wx["humidity_avg_prev24h"] = wx["humidity"].shift(1).rolling(24, min_periods=6).mean()
wx["cloud_cover_avg_prev24h"] = wx["cloud_cover"].shift(1).rolling(24, min_periods=6).mean()
wx["comfort_index_avg_prev24h"] = wx["comfort_index"].shift(1).rolling(24, min_periods=6).mean()

# previous 72h precipitation totals / means
wx["precip_mm_sum_prev72h"] = wx["precip_mm"].shift(1).rolling(72, min_periods=12).sum()
wx["rain_mm_sum_prev72h"] = wx["rain_mm"].shift(1).rolling(72, min_periods=12).sum()

# recent rain flags
wx["had_rain_prev24h"] = (wx["rain_mm"].shift(1).rolling(24, min_periods=6).sum() > 0).astype(float)
wx["had_rain_prev72h"] = (wx["rain_mm"].shift(1).rolling(72, min_periods=12).sum() > 0).astype(float)

games["forecast_origin_time"] = games["exact_time"] - pd.to_timedelta(FORECAST_LEAD_DAYS, unit="D")

# Merge actual game-time weather if you want it for diagnostics only
actual_cols = [
    "wx_time",
    "temperature_f",
    "humidity",
    "wind_mph",
    "precip_mm",
    "rain_mm",
    "snowfall_cm",
    "cloud_cover",
    "weather_condition",
    "precipitation_type",
    "comfort_index"
]

games = pd.merge_asof(
    games.sort_values("exact_time"),
    wx[actual_cols].sort_values("wx_time"),
    left_on="exact_time",
    right_on="wx_time",
    direction="nearest",
    tolerance=pd.Timedelta("1H")
)

games = games.rename(columns={
    "temperature_f": "weather_actual_temperature",
    "humidity": "weather_actual_humidity",
    "wind_mph": "weather_actual_wind_speed",
    "precip_mm": "weather_actual_precipitation",
    "rain_mm": "weather_actual_rain_mm",
    "snowfall_cm": "weather_actual_snowfall_cm",
    "cloud_cover": "weather_actual_cloud_cover",
    "weather_condition": "weather_actual_condition",
    "precipitation_type": "weather_actual_precipitation_type",
    "comfort_index": "weather_actual_comfort_index",
    "wx_time": "weather_actual_time"
})

# Merge proxy weather available at forecast origin
proxy_cols = [
    "wx_time",
    "temperature_f",
    "humidity",
    "wind_mph",
    "precip_mm",
    "rain_mm",
    "snowfall_cm",
    "cloud_cover",
    "weather_condition",
    "precipitation_type",
    "comfort_index",
    "temp_f_avg_prev24h",
    "wind_mph_avg_prev24h",
    "humidity_avg_prev24h",
    "cloud_cover_avg_prev24h",
    "comfort_index_avg_prev24h",
    "precip_mm_sum_prev72h",
    "rain_mm_sum_prev72h",
    "had_rain_prev24h",
    "had_rain_prev72h"
]

games = pd.merge_asof(
    games.sort_values("forecast_origin_time"),
    wx[proxy_cols].sort_values("wx_time"),
    left_on="forecast_origin_time",
    right_on="wx_time",
    direction="nearest",
    tolerance=pd.Timedelta("1H")
)

games = games.rename(columns={
    "wx_time": f"{PREFIX}_time",
    "temperature_f": f"{PREFIX}_temperature",
    "humidity": f"{PREFIX}_humidity",
    "wind_mph": f"{PREFIX}_wind_speed",
    "precip_mm": f"{PREFIX}_precipitation",
    "rain_mm": f"{PREFIX}_rain_mm",
    "snowfall_cm": f"{PREFIX}_snowfall_cm",
    "cloud_cover": f"{PREFIX}_cloud_cover",
    "weather_condition": f"{PREFIX}_weather_condition",
    "precipitation_type": f"{PREFIX}_precipitation_type",
    "comfort_index": f"{PREFIX}_comfort_index",
    "temp_f_avg_prev24h": f"{PREFIX}_temp_avg_prev24h",
    "wind_mph_avg_prev24h": f"{PREFIX}_wind_avg_prev24h",
    "humidity_avg_prev24h": f"{PREFIX}_humidity_avg_prev24h",
    "cloud_cover_avg_prev24h": f"{PREFIX}_cloud_cover_avg_prev24h",
    "comfort_index_avg_prev24h": f"{PREFIX}_comfort_avg_prev24h",
    "precip_mm_sum_prev72h": f"{PREFIX}_precip_sum_prev72h",
    "rain_mm_sum_prev72h": f"{PREFIX}_rain_sum_prev72h",
    "had_rain_prev24h": f"{PREFIX}_had_rain_prev24h",
    "had_rain_prev72h": f"{PREFIX}_had_rain_prev72h"
})

games[f"{PREFIX}_lead_days"] = FORECAST_LEAD_DAYS

# Keep only new cols to merge back
new_weather_cols = [
    # f"{PREFIX}_time",
    f"{PREFIX}_temperature",
    f"{PREFIX}_humidity",
    f"{PREFIX}_wind_speed",
    f"{PREFIX}_precipitation",
    # f"{PREFIX}_rain_mm",
    # f"{PREFIX}_snowfall_cm",
    f"{PREFIX}_cloud_cover",
    f"{PREFIX}_weather_condition",
    # f"{PREFIX}_precipitation_type",
    f"{PREFIX}_comfort_index",
    f"{PREFIX}_temp_avg_prev24h",
    # f"{PREFIX}_wind_avg_prev24h",
    # f"{PREFIX}_humidity_avg_prev24h",
    # f"{PREFIX}_cloud_cover_avg_prev24h",
    # f"{PREFIX}_comfort_avg_prev24h",
    # f"{PREFIX}_precip_sum_prev72h",
    # f"{PREFIX}_rain_sum_prev72h",
    f"{PREFIX}_had_rain_prev24h",
    # f"{PREFIX}_had_rain_prev72h",
    # f"{PREFIX}_lead_days",
]

# Actual weather (for checks, not modeling)
actual_weather_cols = [
    "weather_actual_time",
    "weather_actual_temperature",
    "weather_actual_humidity",
    "weather_actual_wind_speed",
    "weather_actual_precipitation",
    "weather_actual_rain_mm",
    "weather_actual_snowfall_cm",
    "weather_actual_cloud_cover",
    "weather_actual_condition",
    "weather_actual_precipitation_type",
    "weather_actual_comfort_index",
]

out = games[["exact_time"] + new_weather_cols + actual_weather_cols].copy()

df = df.drop(columns=new_weather_cols + actual_weather_cols, errors="ignore")
df = df.merge(out, on="exact_time", how="left")

/var/folders/pt/y9n6c10x2kz3_ty0xm4p0gnm0000gn/T/ipykernel_37174/3318047886.py:171: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  tolerance=pd.Timedelta("1H")
/var/folders/pt/y9n6c10x2kz3_ty0xm4p0gnm0000gn/T/ipykernel_37174/3318047886.py:218: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  tolerance=pd.Timedelta("1H")


In [24]:
### DROP ACTUAL WEATHER COLUMNS (keep only forecast proxy)
df = df.drop(columns=[c for c in actual_weather_cols if c in df.columns])

In [25]:
### RIVALRY FLAG
rivals = {"Duke", "NC State", "East Carolina", "Wake Forest", "Florida State", "Georgia Tech"}
df["rivalry"] = df["opponent"].isin(rivals).astype(int)

In [26]:
df["opponent"].value_counts().head(10)

opponent
Seton Hall          43
Duke                43
Virginia            41
Georgia Tech        38
Miami               36
NC State            35
Wake Forest         33
Coastal Carolina    28
Florida State       25
East Carolina       25
Name: count, dtype: int64

In [27]:
def time_bucket(dt):
    if pd.isna(dt):
        return "unknown"
    h = dt.hour
    if h < 14:
        return "day"
    elif h < 18:
        return "afternoon"
    else:
        return "night"

df["time_bucket"] = df["exact_time"].apply(time_bucket)

In [28]:
### WEEKEND AND MONTH FLAGS
df["weekend"] = (df["exact_time"].dt.weekday >= 4).astype(int)
df["month"] = df["exact_time"].dt.month

In [29]:
### EARLY SEASON, MOMENTUM, PLAYOFF, AND EXCITEMENT
df["early_season"] = df["month"].isin([2, 3]).astype(int)

# momentum index
df["momentum_index"] = (
    0.33 * (df["last3games"] / 3) +
    0.34 * (df["last5games"] / 5) +
    0.33 * df["winlast"]
)

# playoff flag from season_stage
df["playoff"] = df["season_stage"].isin(["regional", "super_regional"]).astype(int)

# excitement index = momentum × standardized rest
rest = df["days_since_last_hg"].astype(float)
rest_z = (rest - rest.mean()) / rest.std(ddof=0)
df["excitement_index"] = df["momentum_index"] * rest_z

In [30]:
### SEASON YEAR
df["season_year"] = pd.to_datetime(df["exact_time"], errors="coerce").dt.year
df = df.drop(columns=["season"])

In [31]:
### VENUE CAPACITY AND ATTENDANCE FLAGS
# boshamer: 4100 (5000 standing) after 2008 renovation, 3000 before
# 2008 played at USA Baseball NTC in Cary (excluded from home games)

df["venue_cap"] = np.where(df["season_year"] < 2008, 3000, 4100)

# occupancy rate
df["occupancy"] = df["attendance"] / df["venue_cap"]

# sold out and low attendance thresholds

sold_thresh = np.where(df["season_year"] < 2008, 2850, 3900)

low_thresh  = np.where(df["season_year"] < 2008,  900, 1250)

df["sold_out"] = (df["attendance"] > sold_thresh).astype(int)
df["low_attendance"] = (df["attendance"] < low_thresh).astype(int)

In [32]:
### PLAYOFF SERIES CLEANUP
# playoff games aren't part of regular series so zero those out
playoff_mask = df["playoff"].fillna(0).astype(int).eq(1)
df.loc[playoff_mask, ["series", "game_in_series"]] = 0

In [33]:
### PREVIOUS ATTENDANCE AND SEASON AVERAGES
att = df[["season_year", "attendance"]].copy()
att["prev_attendance"] = att.groupby("season_year")["attendance"].shift(1)

# season-level averages
year_mean = (
    df.groupby("season_year")["attendance"]
    .mean()
    .sort_index()
)

# exclude covid seasons
valid_year_mean = year_mean.drop([2020, 2021], errors="ignore")

# previous valid season mean
prev_season_mean = valid_year_mean.shift(1)

# rolling 3-year average
prev3yr_mean = valid_year_mean.rolling(3, min_periods=1).mean().shift(1)

# year-over-year change vs prior season).
attendance_yoy_change = valid_year_mean.shift(1) - valid_year_mean.shift(2)

# Fill first-game NaNs with prior-years mean
att["prev_attendance"] = att["prev_attendance"].fillna(
    att["season_year"].map(prev_season_mean)
)

# Add features to df
df["prev_attendance"] = att["prev_attendance"]
df["prev_season_attendance"] = df["season_year"].map(prev_season_mean)
df["prev_3yr_attendance"] = df["season_year"].map(prev3yr_mean)
df["attendance_yoy_change"] = df["season_year"].map(attendance_yoy_change)


In [34]:
### ROLLING ATTENDANCE AVERAGES
df = df.sort_values(["season_year", "game_number"])

# last 3 and last 5 game averages (shifted so current game isn't included)
df["att_avg_last3"] = (
    df.groupby("season_year")["attendance"]
    .shift(1)
    .rolling(3, min_periods=1)
    .mean()
)

df["att_avg_last5"] = (
    df.groupby("season_year")["attendance"]
    .shift(1)
    .rolling(5, min_periods=1)
    .mean()
)

# running season average up to (but not including) current game
df["att_avg_season_to_date"] = (
    df.groupby("season_year")["attendance"]
    .shift(1)
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)

In [35]:
### LAG SERIES ATTENDANCE
df["lag_series_attendance"] = df["prev_attendance"].where(df["game_in_series"] > 1)

series_mask = df["series"].fillna(0).astype(int).eq(0)
df.loc[series_mask, ["lag_series_attendance"]] = np.nan

series_mask = df["game_in_series"].fillna(0).astype(int).eq(1)
df.loc[series_mask, ["lag_series_attendance"]] = np.nan

In [36]:
### RENAME FEATURE COLUMNS TO FINAL NAMES
df = df.rename(columns={
    "last3games": "wins_last3",
    "last5games": "wins_last5",
    "winlast": "win_last",
    "days_since_last_home_game": "days_since_last_hg"
})

In [37]:
### DOUBLEHEADER DETECTION
# count how many games are on the same day
df['games_that_day'] = df.groupby('date')['date'].transform('count')

# which game of the doubleheader is this (1st or 2nd)
df['dh_game_number'] = (
    df.sort_values(['date', 'exact_time'])
      .groupby('date')
      .cumcount() + 1
)

# flag: is this game part of a doubleheader?
df['doubleheader'] = (df['games_that_day'] > 1).astype(int)
df.loc[df['doubleheader'] == 0, 'dh_game_number'] = 0

df = df.drop(columns=["games_that_day"])

### Academic Calendar & Enrollment

In [38]:
cal = pd.read_csv(ACADEMIC_CALENDAR_CSV, parse_dates=["Date"])
cal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4018 entries, 0 to 4017
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Year                   4018 non-null   int64         
 1   Date                   4018 non-null   datetime64[ns]
 2   DateDescription        4018 non-null   object        
 3   DayLong                4018 non-null   object        
 4   DayShort               4018 non-null   object        
 5   MonthLong              4018 non-null   object        
 6   MonthShort             4018 non-null   object        
 7   CalendarDay            4018 non-null   int64         
 8   CalendarWeek           4018 non-null   int64         
 9   CalendarWeekStartDate  4018 non-null   object        
 10  CalendarWeekEndDate    4018 non-null   object        
 11  CalendarDayInWeek      4018 non-null   int64         
 12  CalendarMonth          4018 non-null   int64         
 13  Cal

In [39]:
### ACADEMIC CALENDAR CLEANUP
for c in ["SEMESTER", "IN SESH", "BREAK FLG", "BREAK TYPE", "EXAM FLG", "COMMENCEMENT"]:
    cal[c] = cal[c].astype("string").str.strip()

_semester_u = cal["SEMESTER"].str.upper()
_break_flg_u = cal["BREAK FLG"].str.upper()
_break_type_u = cal["BREAK TYPE"].str.upper()
_exam_flg_u = cal["EXAM FLG"].str.upper()
_commence_u = cal["COMMENCEMENT"].str.upper()

# commencement overrides break type
cal.loc[_commence_u.eq("Y"), "BREAK TYPE"] = "COMMENCEMENT"
_break_type_u = cal["BREAK TYPE"].str.upper()  # refresh after overwrite

# base flags
acad_break_raw = _break_flg_u.eq("Y") | cal["BREAK TYPE"].notna()
acad_exams_raw = _exam_flg_u.eq("Y")

# reading day: exams yes, break no
is_reading = _break_type_u.eq("READING DAY")

cal["acad_exams"] = (acad_exams_raw | is_reading).fillna(False)
cal["acad_break"] = (acad_break_raw & ~is_reading).fillna(False)

# semester break = always break
is_sem_break = _break_type_u.eq("SEMESTER BREAK")
cal.loc[is_sem_break, "acad_break"] = True

# semester category
cal["acad_semester"] = _semester_u.replace({"SUMMER 1": "Summer 1", "SUMMER 2": "Summer 2",
                                            "SPRING": "Spring", "FALL": "Fall"})
cal.loc[_semester_u.isna(), "acad_semester"] = pd.NA

# session active: semester present OR exams (even if IN SESH stops)
cal["acad_sesh_active"] = (cal["acad_semester"].notna() | cal["acad_exams"]).fillna(False)

# break type
cal["acad_break_type"] = cal["BREAK TYPE"]

# 0/1 flags
cal["acad_exams"] = cal["acad_exams"].astype("int8")
cal["acad_break"] = cal["acad_break"].astype("int8")
cal["acad_sesh_active"] = cal["acad_sesh_active"].astype("int8")



In [40]:
enr = pd.read_csv(ENROLLMENT_CSV)
enr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   YEAR                    41 non-null     int64 
 1   TERM                    41 non-null     object
 2   UNDERGRAD_HEADCOUNT     41 non-null     int64 
 3   GRADUATE_HEADCOUNT      41 non-null     int64 
 4   PROFESSIONAL_HEADCOUNT  41 non-null     int64 
 5   TOTAL_HEADCOUNT         41 non-null     int64 
 6   FEMALE_HEADCOUNT        41 non-null     int64 
 7   MALE_HEADCOUNT          41 non-null     int64 
dtypes: int64(7), object(1)
memory usage: 2.7+ KB


In [41]:
### MAP TERM TO CALENDAR SEMESTER LABELS
term_map = {
    "SPRING": "Spring",
    "FALL": "Fall",
    "SUMM_1": "Summer 1",
    "SUMM_2": "Summer 2",
}
enr["acad_semester"] = (
    enr["TERM"].astype("string").str.strip().str.upper().map(term_map).astype("string")
)

# --- rename enrollment columns to what you want ---
enr2 = enr.rename(columns={
    "YEAR": "acad_year",
    "UNDERGRAD_HEADCOUNT": "undergrad_enr",
    "GRADUATE_HEADCOUNT": "graduate_enr",
    "PROFESSIONAL_HEADCOUNT": "professional_enr",
    "MALE_HEADCOUNT": "male_enr",
    "FEMALE_HEADCOUNT": "female_enr",
}).copy()

# total_enr (you can trust source or recompute; I recompute as a check)
enr2["total_enr"] = (
    enr2["undergrad_enr"] + enr2["graduate_enr"] + enr2["professional_enr"]
).astype(int)

enr2 = enr2[[
    "acad_year", "acad_semester",
    "undergrad_enr", "graduate_enr", "professional_enr",
    "male_enr", "female_enr", "total_enr"
]]

# --- prep cal join keys ---
cal["acad_year"] = cal["Year"].astype(int)

# --- merge enrollment into calendar (many dates -> one term row) ---
cal = cal.merge(
    enr2,
    on=["acad_year", "acad_semester"],
    how="left",
    validate="many_to_one"
)



In [42]:
### MERGE ACADEMIC CALENDAR ONTO GAME DATA
# normalize both date columns to midnight so the merge works cleanly
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
cal["Date"] = pd.to_datetime(cal["Date"]).dt.normalize()

# merge on date
df = df.merge(
    cal[[
        "Date",
        "acad_semester",
        "acad_sesh_active",
        "acad_break",
        "acad_exams",
        "acad_break_type",
        "undergrad_enr",
        "graduate_enr",
        "professional_enr",
        "male_enr",
        "female_enr",
        "total_enr"
    ]],
    left_on="date",
    right_on="Date",
    how="left",
    validate="many_to_one"
)

# cleanup
df = df.drop(columns=["Date"])


In [43]:
### SCHOOL HOURS AND SCHOOL NIGHT FLAGS

weekday = df["exact_time"].dt.weekday        # 0=Mon, 6=Sun
next_day = (weekday + 1) % 7

# if we don't know the session status, assume active (conservative)
active = df["acad_sesh_active"].fillna(1)

# school_hours: weekday daytime game during active session
df["school_hours"] = (
    (weekday < 5) &
    (df["time_bucket"].isin(["morning","midday","afternoon"])) &
    (active == 1)
).astype("int8")

# school_night: evening game where tomorrow is a school day
df["school_night"] = (
    (df["time_bucket"] == "night") &
    (next_day < 5) &
    (active == 1)
).astype("int8")


### Google Trends

In [44]:
web = pd.read_csv(GTRENDS_WEB_CSV, parse_dates=["Time"])
web.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 12 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Time                               267 non-null    datetime64[ns]
 1   YEAR                               267 non-null    int64         
 2   MONTH                              267 non-null    int64         
 3   North Carolina Tar Heels baseball  267 non-null    int64         
 4   unc baseball                       267 non-null    int64         
 5   carolina baseball                  267 non-null    int64         
 6   north carolina baseball            267 non-null    int64         
 7   unc baseball schedule              267 non-null    int64         
 8   unc baseball score                 267 non-null    int64         
 9   unc baseball roster                267 non-null    int64         
 10  tar heels baseball                 267

In [45]:
yt = pd.read_csv(GTRENDS_YT_CSV, parse_dates=["Time"])
yt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 12 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Time                               219 non-null    datetime64[ns]
 1   YEAR                               219 non-null    int64         
 2   MONTH                              219 non-null    int64         
 3   North Carolina Tar Heels baseball  219 non-null    int64         
 4   unc baseball                       219 non-null    int64         
 5   carolina baseball                  219 non-null    int64         
 6   north carolina baseball            219 non-null    int64         
 7   unc baseball schedule              219 non-null    int64         
 8   unc baseball score                 219 non-null    int64         
 9   unc baseball roster                219 non-null    int64         
 10  tar heels baseball                 219

In [46]:
### GT MERGE (DAY-BEFORE SAFE WITH MONTHLY TOTALS)

### build monthly GT lookup tables using actual calendar month
web_monthly = (
    web[["Time", "TOTAL_QUERIES"]]
      .copy()
      .assign(gt_period=lambda x: x["Time"].dt.to_period("M"))
      .drop_duplicates(subset=["gt_period"])
      .rename(columns={"TOTAL_QUERIES": "gt_web"})
      [["gt_period", "gt_web"]]
)

yt_monthly = (
    yt[["Time", "TOTAL_QUERIES"]]
      .copy()
      .assign(gt_period=lambda x: x["Time"].dt.to_period("M"))
      .drop_duplicates(subset=["gt_period"])
      .rename(columns={"TOTAL_QUERIES": "gt_youtube"})
      [["gt_period", "gt_youtube"]]
)

df["gt_period_d1"] = (pd.to_datetime(df["date"]) - pd.Timedelta(days=1)).dt.to_period("M") - 1

### merge day-before-safe monthly GT onto each game row
df = df.merge(
    web_monthly,
    left_on="gt_period_d1",
    right_on="gt_period",
    how="left",
    validate="many_to_one"
).drop(columns=["gt_period"])

df = df.merge(
    yt_monthly,
    left_on="gt_period_d1",
    right_on="gt_period",
    how="left",
    validate="many_to_one"
).drop(columns=["gt_period"])

### total monthly GT used in the day-before model
df["gt_total"] = df["gt_web"] + df["gt_youtube"]

### build preseason GT = January + February totals for each season
web_preseason = (
    web.loc[web["MONTH"].isin([1, 2])]
       .groupby("YEAR", as_index=False)["TOTAL_QUERIES"]
       .sum()
       .rename(columns={
           "YEAR": "season_year",
           "TOTAL_QUERIES": "gt_web_preseason"
       })
)

yt_preseason = (
    yt.loc[yt["MONTH"].isin([1, 2])]
      .groupby("YEAR", as_index=False)["TOTAL_QUERIES"]
      .sum()
      .rename(columns={
          "YEAR": "season_year",
          "TOTAL_QUERIES": "gt_youtube_preseason"
      })
)

### merge preseason GT onto every game in that season
df = df.merge(web_preseason, on="season_year", how="left", validate="many_to_one")
df = df.merge(yt_preseason,  on="season_year", how="left", validate="many_to_one")

### total preseason GT
df["gt_total_preseason"] = df["gt_web_preseason"] + df["gt_youtube_preseason"]

### UNC Basketball

In [47]:
### LOAD UNC BASKETBALL SCHEDULE (for MBB overlap features)
basket = pd.read_csv(UNC_BASKETBALL_CSV)

basket.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 926 entries, 0 to 925
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   season          926 non-null    object 
 1   game_number     926 non-null    int64  
 2   date            926 non-null    object 
 3   start_time      926 non-null    object 
 4   opponent        926 non-null    object 
 5   tournament      249 non-null    object 
 6   venue           926 non-null    object 
 7   result          926 non-null    object 
 8   unc_score       926 non-null    int64  
 9   opponent_score  926 non-null    int64  
 10  attendance      831 non-null    float64
dtypes: float64(1), int64(3), object(7)
memory usage: 79.7+ KB


In [48]:
### PARSE DATES

basket["date_clean"] = basket["date"].str.replace(r"\(.*\)", "", regex=True).str.strip()

def parse_basket_date(row):
    month_str = row["date_clean"].split()[0]

    year_start = int(row["season"][:4])
    year_end = year_start + 1

    # months that belong to NEXT calendar year
    if month_str in ["Jan", "Feb", "Mar", "Apr"]:
        year = year_end
    else:
        year = year_start

    return f"{row['date_clean']} {year}"

basket["date_full"] = basket.apply(parse_basket_date, axis=1)

basket["date_norm"] = pd.to_datetime(
    basket["date_full"],
    format="%b %d %Y",
    errors="coerce"
)

In [49]:
### GAME-LEVEL BASKETBALL INDICATORS

basket["mbb_game"] = np.where(
    basket["venue"].str.contains("Dean E. Smith", case=False, na=False),
    "Home",
    "Away"
)

basket["mbb_tourney"] = basket["tournament"].isin(
    ["ACC Tournament", "NCAA Tournament"]
).astype(int)

### make sure merge keys are same dtype
basket["date_norm"] = pd.to_datetime(basket["date_norm"], errors="coerce").dt.normalize()
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

### COLLAPSE BASKETBALL TO DAY LEVEL
basket_day = (
    basket.groupby("date_norm", as_index=False)
    .agg(
        mbb_game=("mbb_game", lambda x: "Home" if "Home" in x.values else "Away"),
        mbb_tourney=("mbb_tourney", "max")
    )
)

### MERGE BASKETBALL DATA ONTO BASEBALL
df = df.merge(
    basket_day,
    left_on="date",
    right_on="date_norm",
    how="left"
)

### any missing value after merge means no basketball game that day
df["mbb_game"] = df["mbb_game"].fillna("None")
df["mbb_tourney"] = df["mbb_tourney"].fillna(0).astype(int)

### create binary flags
df["mbb_any"] = df["mbb_game"].isin(["Home", "Away"]).astype(int)
df["mbb_home"] = (df["mbb_game"] == "Home").astype(int)
df["mbb_away"] = (df["mbb_game"] == "Away").astype(int)

df = df.drop(columns=["mbb_game", "date_norm"], errors="ignore")

### Promotions

In [50]:
### PARSE PROMOTION DATA

import re
import numpy as np

# figure out which seasons actually have promo data (some early years don't)
has_promo_data = df.groupby("season_year")["promo_scrape"].transform(lambda x: x.notna().any()).astype(bool)

def clean_promo_text(x):
    if pd.isna(x):
        return x

    x = x.lower()

    ### remove "presented by ..." tails
    x = re.sub(r"presented by[:]?.*", "", x)

    ### remove leading giveaway label but keep the actual item
    x = re.sub(r"^giveaway:\s*", "", x)

    ### normalize separators so count logic is more consistent
    x = re.sub(r"\s*\|\s*", "/", x)
    x = re.sub(r"\s*&\s*", "/", x)

    ### collapse extra whitespace
    x = re.sub(r"\s+", " ", x).strip()

    return x

s = df["promo_scrape"].apply(clean_promo_text)

# binary promo categories
df["promo_giveaway"] = s.str.contains(
    r"bobblehead|magnet|trading card|trading cards|card set|card sets|koozie|keychain|sunglasses|replica jersey|poster|posters|rally towel|towel|easter egg",
    regex=True
).fillna(False)

df["promo_fireworks"] = s.str.contains(
    r"fireworks?",
    regex=True
).fillna(False)

df["promo_kids"] = s.str.contains(
    r"kids|run the bases|youth",
    regex=True
).fillna(False)

df["promo_bark"] = s.str.contains(
    r"bark at the bosh",
    regex=True
).fillna(False)

df["promo_discount"] = s.str.contains(
    r"dollar dog|\$1 dog|two for tuesday|wing wednesday",
    regex=True
).fillna(False)

df["promo_theme"] = s.str.contains(
    r"blue out|halloween|2000s|80\.?s day|appreciation|first responders|salute to service|salute|fan appreciation|senior day|awareness|spirit night|spirit day|shoes4hope|players club|opening day|mardi gras|st\. patrick|vs\.? cancer|military|teacher",
    regex=True
).fillna(False)

df["promo_trivia"] = s.str.contains(
    r"trivia",
    regex=True
).fillna(False)

promo_cols = [
    "promo_giveaway", "promo_fireworks", "promo_kids", "promo_bark",
    "promo_discount", "promo_theme", "promo_trivia",
]

# count how many promos each game has (separated by / in cleaned text)
df["promo_count"] = np.nan
df.loc[s.notna(), "promo_count"] = s.str.count(r"/") + 1

# promo text exists but none of the tracked buckets matched
df["promo_other"] = (
    s.notna() &
    (df[promo_cols].sum(axis=1) == 0)
)

# fill seasons with known promo data but missing game-level promo as 0
df.loc[has_promo_data, promo_cols + ["promo_other"]] = df.loc[has_promo_data, promo_cols + ["promo_other"]].fillna(0)
df.loc[has_promo_data & df["promo_scrape"].isna(), "promo_count"] = 0

# seasons without promo data get NaN (not 0) so the model knows it's missing
df.loc[~has_promo_data, promo_cols + ["promo_other", "promo_count"]] = np.nan

# cast to float so they survive numeric operations
for c in promo_cols + ["promo_other"]:
    df[c] = df[c].astype(float)

print(f"Promo features created: {len(promo_cols) + 2} columns")
print(f"  Seasons with promo data: {df[has_promo_data]['season_year'].nunique()}")
print(f"  Uncategorized promo rows (promo_other=1): {int((df['promo_other'] == 1).sum())}")

Promo features created: 9 columns
  Seasons with promo data: 7
  Uncategorized promo rows (promo_other=1): 9


/var/folders/pt/y9n6c10x2kz3_ty0xm4p0gnm0000gn/T/ipykernel_37174/3397522448.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False)
/var/folders/pt/y9n6c10x2kz3_ty0xm4p0gnm0000gn/T/ipykernel_37174/3397522448.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False)
/var/folders/pt/y9n6c10x2kz3_ty0xm4p0gnm0000gn/T/ipykernel_37174/3397522448.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to th

In [51]:
### DROP RAW PROMO COLUMNS
df = df.drop(
    columns=["promo", "promo_scrape", "big_promo", "small_promo"],
    errors="ignore"
)

### Performance Stats (L10 - not used)


In [52]:
### L10 ROLLING PERFORMANCE STATS (not used)
# columns: unc_bat_hr_pg_l10, unc_pit_ra_pg_l10, opp_bat_hr_pg_l10, etc.
# uncomment the merge below to include them

# perf_small = perf[["season", "game_number", ...]]
# df = df.merge(perf_small, left_on=["season_year","game_number"], right_on=["season","game_number"], how="left")
# df = df.drop(columns=["season"])

### Performance Stats (Per Game)

In [53]:
perf2 = pd.read_csv(PERFORMANCE_CSV)

perf2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 98 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   season                           498 non-null    int64  
 1   game_date                        498 non-null    object 
 2   dh_index                         498 non-null    int64  
 3   opponent                         498 non-null    object 
 4   game_number                      498 non-null    int64  
 5   date                             498 non-null    object 
 6   start_time                       498 non-null    object 
 7   tournament                       95 non-null     object 
 8   venue                            498 non-null    object 
 9   promotion                        129 non-null    object 
 10  result                           486 non-null    object 
 11  unc_score                        486 non-null    float64
 12  opponent_score        

In [54]:
### PERFORMANCE COLUMNS TO MERGE FROM V11
perf2_cols = [
    "season", "game_number",

    # current season team stats
    "runs_pg", "hr_pg", "errors_pg", "idp_pg", "run_diff_pg",

    # current season opponent stats
    "opp_runs_pg", "opp_hr_pg", "opp_errors_pg", "opp_run_diff_pg",
    "opp_win_pct", "opp_wins_last3", "opp_wins_last5",

    # matchup
    "strength_diff", "win_pct_diff",

    # prev season team stats
    "runs_pg_prev_season", "runs_allowed_pg_prev_season",
    "hr_pg_prev_season", "errors_pg_prev_season",
    "k_pg_prev_season", "so_pg_prev_season",
    "hr_allowed_pg_prev_season", "idp_pg_prev_season",
    "run_diff_pg_prev_season", "win_pct_prev_season",
    "team_strength_prev_season",

    # prev season opponent stats
    "opp_runs_pg_prev_season", "opp_runs_allowed_pg_prev_season",
    "opp_hr_pg_prev_season", "opp_errors_pg_prev_season",
    "opp_k_pg_prev_season", "opp_so_pg_prev_season",
    "opp_hr_allowed_pg_prev_season", "opp_idp_pg_prev_season",
    "opp_run_diff_pg_prev_season", "opp_win_pct_prev_season",
    "opp_team_strength_prev_season",

    # flag
    "has_opponent_features",
]

perf2_small = perf2[perf2_cols]

In [55]:
### MERGE PERFORMANCE STATS ONTO GAME DATA
# join on season + game_number so each game gets its rolling stats
df = df.merge(
    perf2_small,
    left_on=["season_year", "game_number"],
    right_on=["season", "game_number"],
    how="left"
)
df = df.drop(columns=["season"])

In [56]:
### PERFORMANCE FEATURES NOTE
# opp_win_pct, run_diff_pg, and opp_run_diff_pg now come directly
# from uncbaseball_season_performance_0330.csv, no need to derive them manually
print(f"Performance columns merged: {[c for c in df.columns if c in perf2_cols]}")

Performance columns merged: ['game_number', 'runs_pg', 'hr_pg', 'errors_pg', 'idp_pg', 'run_diff_pg', 'opp_runs_pg', 'opp_hr_pg', 'opp_errors_pg', 'opp_run_diff_pg', 'opp_win_pct', 'opp_wins_last3', 'opp_wins_last5', 'strength_diff', 'win_pct_diff', 'runs_pg_prev_season', 'runs_allowed_pg_prev_season', 'hr_pg_prev_season', 'errors_pg_prev_season', 'k_pg_prev_season', 'so_pg_prev_season', 'hr_allowed_pg_prev_season', 'idp_pg_prev_season', 'run_diff_pg_prev_season', 'win_pct_prev_season', 'team_strength_prev_season', 'opp_runs_pg_prev_season', 'opp_runs_allowed_pg_prev_season', 'opp_hr_pg_prev_season', 'opp_errors_pg_prev_season', 'opp_k_pg_prev_season', 'opp_so_pg_prev_season', 'opp_hr_allowed_pg_prev_season', 'opp_idp_pg_prev_season', 'opp_run_diff_pg_prev_season', 'opp_win_pct_prev_season', 'opp_team_strength_prev_season', 'has_opponent_features']


In [57]:
### DERIVE PREV-SEASON MATCHUP DIFFERENTIALS
# these are knowable preseason: team vs opponent prev season stats

if 'team_strength_prev_season' in df.columns and 'opp_team_strength_prev_season' in df.columns:
    df['strength_diff_prev_season'] = df['team_strength_prev_season'] - df['opp_team_strength_prev_season']
    print(f"strength_diff_prev_season: {df['strength_diff_prev_season'].notna().sum()} non-null")

if 'win_pct_prev_season' in df.columns and 'opp_win_pct_prev_season' in df.columns:
    df['win_pct_diff_prev_season'] = df['win_pct_prev_season'] - df['opp_win_pct_prev_season']
    print(f"win_pct_diff_prev_season: {df['win_pct_diff_prev_season'].notna().sum()} non-null")

strength_diff_prev_season: 449 non-null
win_pct_diff_prev_season: 449 non-null


### Opponent distance traveled

In [58]:
coords = pd.read_csv(TEAM_HOME_COORDS_BASEBALL_CSV)

coords.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1079 entries, 0 to 1078
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   season     1079 non-null   int64  
 1   opponent   1079 non-null   object 
 2   venue      1079 non-null   object 
 3   latitude   1079 non-null   float64
 4   longitude  1079 non-null   float64
dtypes: float64(2), int64(1), object(2)
memory usage: 42.3+ KB


In [59]:
### PREPARE OPPONENT COORDINATES FOR DISTANCE CALCULATION
coords2 = coords.rename(columns={
    "season": "season_year",
    "latitude": "opp_lat",
    "longitude": "opp_lon"
})[["season_year", "opponent", "opp_lat", "opp_lon"]].copy()

In [60]:
### MERGE OPPONENT COORDINATES
df = df.merge(
    coords2,
    on=["season_year", "opponent"],
    how="left"
)

In [61]:
### CALCULATE OPPONENT TRAVEL DISTANCE
UNC_LAT, UNC_LON = 35.9115, -79.0528

def haversine_miles(lat1, lon1, lat2, lon2):
    """great circle distance between two points in miles"""
    R = 3958.7613  # earth radius in miles

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

# calculate how far the opponent had to travel to get to Chapel Hill
df["opp_distance_miles"] = haversine_miles(
    UNC_LAT, UNC_LON,
    df["opp_lat"],
    df["opp_lon"]
)

# don't need the raw coordinates anymore
df = df.drop(columns=["opp_lat", "opp_lon"])

### Re-order and download

In [62]:
### RE-ORDER COLUMNS

# event metadata
event_cols = [
    "event_id", "date", "time", "exact_time", "season_year", "season_stage",
    "win", "unc_score", "opp_score",
]

# opponent info
opponent_cols = [
    "opponent", "acc", "rivalry", "opp_distance_miles", "opponent_grp"
]

# scheduling features
schedule_cols = [
    "opening_day", "game_number", "time_bucket", "day_of_week", "weekend", "month",
    "days_since_last_hg", "early_season", "playoff",
    "series", "game_in_series", "doubleheader", "dh_game_number",
    "mbb_any", "mbb_home", "mbb_away", "mbb_tourney",
]

# attendance features
attendance_cols = [
    "attendance", "venue_cap", "occupancy", "sold_out", "low_attendance",
    "lag_series_attendance", "prev_attendance", "att_avg_last3", "att_avg_last5",
    "att_avg_season_to_date",
    "prev_season_attendance", "prev_3yr_attendance", "attendance_yoy_change",
]

# performance stats
performance_cols = [
    "record_pct", "prev_szn_record_pct",
    "win_last", "wins_last3", "wins_last5",
    "momentum_index", "excitement_index",
    "runs_pg", "hr_pg", "errors_pg", "idp_pg", "run_diff_pg",
    "opp_runs_pg", "opp_hr_pg", "opp_errors_pg", "opp_run_diff_pg",
    "opp_win_pct", "opp_wins_last3", "opp_wins_last5",
    "strength_diff", "win_pct_diff",
    "runs_pg_prev_season", "runs_allowed_pg_prev_season",
    "hr_pg_prev_season", "errors_pg_prev_season",
    "k_pg_prev_season", "so_pg_prev_season",
    "hr_allowed_pg_prev_season", "idp_pg_prev_season",
    "run_diff_pg_prev_season", "win_pct_prev_season", "team_strength_prev_season",
    "opp_runs_pg_prev_season", "opp_runs_allowed_pg_prev_season",
    "opp_hr_pg_prev_season", "opp_errors_pg_prev_season",
    "opp_k_pg_prev_season", "opp_so_pg_prev_season",
    "opp_hr_allowed_pg_prev_season", "opp_idp_pg_prev_season",
    "opp_run_diff_pg_prev_season", "opp_win_pct_prev_season", "opp_team_strength_prev_season",
    "has_opponent_features",
    "strength_diff_prev_season", "win_pct_diff_prev_season",
]

# weather features
weather_cols = [
    "wx_d2_temperature", "wx_d2_humidity", "wx_d2_wind_speed",
    "wx_d2_precipitation", "wx_d2_cloud_cover",
    "wx_d2_weather_condition", "wx_d2_comfort_index",
    "wx_d2_temp_avg_prev24h", "wx_d2_had_rain_prev24h",
]

# academic calendar + enrollment
academics_cols = [
    "acad_sesh_active", "acad_semester", "acad_break",
    "acad_exams", "acad_break_type", "school_hours", "school_night",
    "undergrad_enr", "graduate_enr", "professional_enr",
    "female_enr", "male_enr", "total_enr",
]

# google trends
social_cols = [
    "gt_web", "gt_youtube", "gt_total",
    "gt_web_preseason", "gt_youtube_preseason", "gt_total_preseason",
]

# promo features
promo_cols_order = [
    "promo_count", "promo_giveaway", "promo_fireworks", "promo_kids",
    "promo_bark", "promo_discount", "promo_theme", "promo_trivia", "promo_other",
]

COL_ORDER = (
    event_cols + opponent_cols + schedule_cols + attendance_cols +
    performance_cols + weather_cols + academics_cols + social_cols + promo_cols_order
)

ordered_existing_cols = [c for c in COL_ORDER if c in df.columns]
remaining_cols = [c for c in df.columns if c not in ordered_existing_cols]
df = df[ordered_existing_cols + remaining_cols]

In [63]:
### PREP FUNCTION FOR PREDICTION TIER
# same feature engineering as prep_tier but does not require attendance

def prep_pred_tier(tier_df):
    working_df = tier_df.copy()

    working_df = working_df[~working_df['season_year'].isin([2020, 2021])].reset_index(drop=True)

    working_df['day_of_week'] = pd.to_datetime(working_df['date']).dt.dayofweek
    working_df['month'] = pd.to_datetime(working_df['date']).dt.month
    working_df['day_of_week_sin'] = np.sin(2 * np.pi * working_df['day_of_week'] / 7)
    working_df['day_of_week_cos'] = np.cos(2 * np.pi * working_df['day_of_week'] / 7)
    working_df['month_sin'] = np.sin(2 * np.pi * working_df['month'] / 12)
    working_df['month_cos'] = np.cos(2 * np.pi * working_df['month'] / 12)

    meta_cols = ['event_id', 'season_year', 'date', 'time', 'exact_time', 'opponent', 'attendance']
    meta = working_df[[col for col in meta_cols if col in working_df.columns]].copy()

    # fill prev_attendance with the most recent earlier actual attendance
    if 'prev_attendance' in working_df.columns:
        fallback_prev_attendance = working_df['attendance'].shift(1)
        working_df['prev_attendance'] = working_df['prev_attendance'].fillna(fallback_prev_attendance)

    drop_cols = [
        'event_id', 'date', 'time', 'exact_time',
        'attendance', 'occupancy', 'sold_out', 'low_attendance',
        'day_of_week', 'month', 'opponent'
    ]

    features = working_df.drop(columns=[col for col in drop_cols if col in working_df.columns])

    if 'lag_series_attendance' in features.columns:
        features['lag_series_attendance'] = features['lag_series_attendance'].fillna(-1)
    if 'prev_szn_record_pct' in features.columns:
        features['prev_szn_record_pct'] = features['prev_szn_record_pct'].fillna(0.5)
    if 'prev_attendance' in features.columns:
        features['prev_attendance'] = features['prev_attendance'].fillna(features['prev_attendance'].median())

    categorical_cols = [col for col in features.columns
                        if features[col].dtype == 'object' or pd.api.types.is_string_dtype(features[col])]
    numeric_cols = [col for col in features.columns if col not in categorical_cols]

    for col in numeric_cols:
        features[col] = features[col].fillna(0)

    for col in categorical_cols:
        features[col] = features[col].fillna("none").astype(str)

    return features, meta


In [64]:
### SCORE 2026 WITH FROZEN DAY-BEFORE ARTIFACT AND UPDATE APP MIDSEASON FILES

def score_2026_with_daybefore_frozen_artifact(source_df, artifact):
    pred_tier = source_df.loc[
        (source_df['season_year'] >= artifact['start_train_year']) &
        (source_df['season_year'] <= artifact['deployment_season']),
        [c for c in artifact['tier_input_cols'] if c in source_df.columns]
    ].copy()

    X_pred_full, meta_pred_full = prep_pred_tier(pred_tier)
    X_pred_full = X_pred_full.reindex(columns=artifact['feature_columns'])

    for col in X_pred_full.columns:
        if pd.api.types.is_string_dtype(X_pred_full[col]) or X_pred_full[col].dtype == 'object':
            X_pred_full[col] = X_pred_full[col].fillna("none").astype(str)
        else:
            X_pred_full[col] = X_pred_full[col].fillna(0)

    target_year = artifact['deployment_season']
    score_mask = meta_pred_full['season_year'] == target_year

    played_mask = score_mask & meta_pred_full['attendance'].notna()
    unplayed_mask = score_mask & meta_pred_full['attendance'].isna()

    # Build a usable game datetime for deciding whether the next unplayed game
    # is within the 1-day scoring window.
    game_dt = pd.to_datetime(meta_pred_full.get('exact_time'), errors='coerce')

    if game_dt.isna().all():
        combined_dt = (
            meta_pred_full['date'].astype(str).fillna('').str.strip()
            + ' '
            + meta_pred_full.get('time', pd.Series('', index=meta_pred_full.index)).astype(str).fillna('').str.strip()
        ).str.strip()
        game_dt = pd.to_datetime(combined_dt, errors='coerce')

    now_ts = pd.Timestamp.now().tz_localize(None)

    next_game_mask = pd.Series(False, index=score_mask.index)

    future_unplayed = meta_pred_full.loc[
        unplayed_mask & game_dt.notna() & (game_dt > now_ts)
    ].copy()

    if not future_unplayed.empty:
        future_unplayed = future_unplayed.assign(_game_dt=game_dt.loc[future_unplayed.index])
        next_idx = future_unplayed.sort_values('_game_dt').index[0]
        next_game_dt = future_unplayed.loc[next_idx, '_game_dt']

        run_date = now_ts.normalize()
        next_game_date = pd.Timestamp(next_game_dt).normalize()

        if next_game_date == (run_date + pd.Timedelta(days=1)):
            next_game_mask.loc[next_idx] = True

    score_mask = played_mask | next_game_mask

    X_score = X_pred_full.loc[score_mask].copy()
    meta_score = meta_pred_full.loc[score_mask].reset_index(drop=True)

    preds = artifact['pipeline'].predict(X_score)

    out = meta_score.copy()
    out['predicted_raw'] = preds
    out['predicted'] = np.round(preds).astype(int)

    out['error'] = np.nan
    out['abs_error'] = np.nan
    out['pct_error'] = np.nan

    scored_mask = out['attendance'].notna()
    metrics_row = None

    if scored_mask.any():
        out.loc[scored_mask, 'error'] = (
            out.loc[scored_mask, 'attendance'] - out.loc[scored_mask, 'predicted_raw']
        )
        out.loc[scored_mask, 'abs_error'] = np.abs(out.loc[scored_mask, 'error'])

        nonzero_mask = scored_mask & (out['attendance'] != 0)
        if nonzero_mask.any():
            out.loc[nonzero_mask, 'pct_error'] = (
                np.abs(out.loc[nonzero_mask, 'attendance'] - out.loc[nonzero_mask, 'predicted_raw'])
                / out.loc[nonzero_mask, 'attendance'] * 100
            )

        scored = out.loc[scored_mask].copy()
        metrics_row = {
            'season': target_year,
            'games': len(scored),
            'rmse': np.sqrt(mean_squared_error(scored['attendance'], scored['predicted_raw'])),
            'mae': mean_absolute_error(scored['attendance'], scored['predicted_raw']),
            'r2': r2_score(scored['attendance'], scored['predicted_raw']),
            'mape': scored['pct_error'].mean(skipna=True),
            'within_500': (scored['abs_error'] <= 500).mean() * 100,
        }

    return out, metrics_row


def replace_season_rows(template_df, new_df, season_col='season_year', target_year=2026):
    template = template_df.copy()
    new = new_df.copy()

    template[season_col] = pd.to_numeric(template[season_col], errors='coerce')
    new[season_col] = pd.to_numeric(new[season_col], errors='coerce')

    keep_old = template.loc[template[season_col] != float(target_year)].copy()

    ordered_cols = list(template.columns) + [c for c in new.columns if c not in template.columns]
    combined = pd.concat([keep_old, new], ignore_index=True, sort=False)
    combined = combined.reindex(columns=ordered_cols)

    sort_cols = [c for c in ['season_year', 'date', 'game_number', 'opponent'] if c in combined.columns]
    if sort_cols:
        combined = combined.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)

    return combined


artifact = joblib.load(DB_ARTIFACT_PATH)
print("loaded artifact:", DB_ARTIFACT_PATH)
print("artifact tier:", artifact['pred_tier_name'])
print("artifact model:", artifact['winning_model'])

db_2026_preds, db_2026_metrics = score_2026_with_daybefore_frozen_artifact(df, artifact)
db_2026_preds['predicted_raw'] = db_2026_preds['predicted_raw'].astype(float)

if db_2026_metrics is not None:
    print(pd.DataFrame([db_2026_metrics]))

db_merge_key = ['event_id', 'season_year', 'date', 'opponent']
db_pred_merge_cols = db_merge_key + ['predicted', 'predicted_raw', 'error', 'abs_error', 'pct_error']

### 2026 FILE 1: full_df slice
db_2026_export_full = df.loc[df['season_year'] == artifact['deployment_season']].merge(
    db_2026_preds[db_pred_merge_cols],
    on=db_merge_key,
    how='inner'
)
db_2026_export_full = db_2026_export_full.rename(columns={'attendance': 'attendance_pred'})
db_front = ['event_id', 'season_year', 'game_number', 'date', 'opponent',
            'attendance_pred', 'predicted', 'error', 'abs_error', 'pct_error']
db_front = [c for c in db_front if c in db_2026_export_full.columns]
db_2026_export_full = db_2026_export_full[db_front + [c for c in db_2026_export_full.columns if c not in db_front]]

### 2026 FILE 2: raw tier slice
db_raw_tier_cols = [c for c in artifact['tier_input_cols'] if c in df.columns]
db_2026_raw_tier_df = df.loc[df['season_year'] == artifact['deployment_season'], db_raw_tier_cols].copy()
db_2026_raw_tier_df = db_2026_raw_tier_df.merge(db_2026_preds[db_pred_merge_cols], on=db_merge_key, how='inner')

db_raw_front = ['event_id', 'season_year', 'game_number', 'date', 'opponent',
                'attendance', 'predicted', 'predicted_raw', 'error', 'abs_error', 'pct_error']
db_raw_front = [c for c in db_raw_front if c in db_2026_raw_tier_df.columns]
db_2026_raw_tier_df = db_2026_raw_tier_df[db_raw_front + [c for c in db_2026_raw_tier_df.columns if c not in db_raw_front]]

### 2026 FILE 3: final features slice
db_reduced_features = [c for c in artifact['feature_columns'] if c in db_2026_raw_tier_df.columns]
db_meta = ['event_id', 'season_year', 'game_number', 'date', 'opponent',
           'attendance', 'predicted', 'predicted_raw', 'error', 'abs_error', 'pct_error']
db_meta = [c for c in db_meta if c in db_2026_raw_tier_df.columns]
db_feat = [c for c in db_reduced_features if c not in db_meta]
db_2026_final_feats_df = db_2026_raw_tier_df[db_meta + db_feat]

### Load templates from the full unified run and replace only the 2026 rows
db_template_full = pd.read_csv(DB_TEMPLATE_FULL)
db_template_raw = pd.read_csv(DB_TEMPLATE_RAW)
db_template_final = pd.read_csv(DB_TEMPLATE_FINAL)

db_export_full = replace_season_rows(db_template_full, db_2026_export_full, target_year=artifact['deployment_season'])
db_raw_tier_df = replace_season_rows(db_template_raw, db_2026_raw_tier_df, target_year=artifact['deployment_season'])
db_final_feats_df = replace_season_rows(db_template_final, db_2026_final_feats_df, target_year=artifact['deployment_season'])

### Write the three midseason files directly into the app data folder
db_export_full.to_csv(APP_DATA_DIR / "db_predictions_2024_2026_full_df.csv", index=False)
db_raw_tier_df.to_csv(APP_DATA_DIR / "db_predictions_2024_2026_tier5_raw_tier_cols.csv", index=False)
db_final_feats_df.to_csv(APP_DATA_DIR / "db_predictions_2024_2026_tier5_final_feats_export.csv", index=False)

print(f"wrote -> {APP_DATA_DIR / 'db_predictions_2024_2026_full_df.csv'} ({db_export_full.shape[0]} rows)")
print(f"wrote -> {APP_DATA_DIR / 'db_predictions_2024_2026_tier5_raw_tier_cols.csv'} ({db_raw_tier_df.shape[0]} rows)")
print(f"wrote -> {APP_DATA_DIR / 'db_predictions_2024_2026_tier5_final_feats_export.csv'} ({db_final_feats_df.shape[0]} rows)")

loaded artifact: /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/SCORING/artifacts/db_frozen_model_for_2026.joblib
artifact tier: tier5
artifact model: elastic_net
   season  games        rmse         mae        r2       mape  within_500
0    2026     25  480.410447  369.828288  0.103955  12.097574        72.0
wrote -> /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/APP/data/db_predictions_2024_2026_full_df.csv (103 rows)
wrote -> /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/APP/data/db_predictions_2024_2026_tier5_raw_tier_cols.csv (103 rows)
wrote -> /Users/danielfulk/GitHub/Personal-S26-TeamA3-App-Repo-/APP/data/db_predictions_2024_2026_tier5_final_feats_export.csv (103 rows)


In [65]:
db_export_full.tail(10)

,event_id,season_year,game_number,date,opponent,attendance_pred,predicted,error,abs_error,pct_error,...,promo_giveaway,promo_fireworks,promo_kids,promo_bark,promo_discount,promo_theme,promo_trivia,promo_other,gt_period_d1,predicted_raw
93,20260310_1500_bucknell,2026.0,18,2026-03-10 00:00:00,Bucknell,2466.0,2170,295.643502,295.643502,11.988788,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2026-02,2170.356498
94,20260318_1500_unc_greensboro,2026.0,22,2026-03-18 00:00:00,UNC Greensboro,2526.0,2191,334.987901,334.987901,13.261595,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2026-02,2191.012099
95,20260320_2000_louisville,2026.0,23,2026-03-20 00:00:00,Louisville,2699.0,3217,-518.465983,518.465983,19.209558,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-02,3217.465983
96,20260321_1400_louisville,2026.0,24,2026-03-21 00:00:00,Louisville,3376.0,3173,202.532474,202.532474,5.999185,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2026-02,3173.467526
97,20260322_1300_louisville,2026.0,25,2026-03-22 00:00:00,Louisville,3231.0,3029,202.095246,202.095246,6.254882,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026-02,3028.904754
98,20260331_2000_campbell,2026.0,30,2026-03-31 00:00:00,Campbell,3140.0,2906,233.754283,233.754283,7.444404,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,2026-02,2906.245717
99,20260402_1800_boston_college,2026.0,31,2026-04-02 00:00:00,Boston College,3031.0,2674,356.579946,356.579946,11.764432,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2026-03,2674.420054
100,20260403_1700_boston_college,2026.0,32,2026-04-03 00:00:00,Boston College,3381.0,3107,273.747208,273.747208,8.096634,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-03,3107.252792
101,20260404_1200_boston_college,2026.0,33,2026-04-04 00:00:00,Boston College,3259.0,3124,134.705265,134.705265,4.133331,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-03,3124.294735
102,20260407_1900_charlotte,2026.0,34,2026-04-07 00:00:00,Charlotte,4289.0,2733,1555.600029,1555.600029,36.269527,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2026-03,2733.399971
